# 03 - Risk Dashboard Analysis
## Polymarket Obfuscation Detection Pipeline

Interactive analysis of risk scores and detection results.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.storage.postgres_store import PostgresStore
from src.analysis.risk_scorer import RiskScorer

In [ ]:
store = PostgresStore()
scorer = RiskScorer(store)

print("Connected to database")

## Risk Distribution Overview

In [ ]:
distribution = scorer.get_risk_distribution()
print("=== Risk Distribution ===")
print(f"Total Wallets: {distribution['total_wallets']}")
print(f"Low Risk (<25): {distribution['low_risk']['count']} ({distribution['low_risk']['percentage']:.1f}%)")
print(f"Medium Risk (25-50): {distribution['medium_risk']['count']} ({distribution['medium_risk']['percentage']:.1f}%)")
print(f"High Risk (50-75): {distribution['high_risk']['count']} ({distribution['high_risk']['percentage']:.1f}%)")
print(f"Critical (>75): {distribution['critical_risk']['count']} ({distribution['critical_risk']['percentage']:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = ['Low\n(<25)', 'Medium\n(25-50)', 'High\n(50-75)', 'Critical\n(>75)']
sizes = [
    distribution['low_risk']['count'],
    distribution['medium_risk']['count'],
    distribution['high_risk']['count'],
    distribution['critical_risk']['count']
]
colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']

axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Risk Distribution')

axes[1].bar(labels, sizes, color=colors)
axes[1].set_title('Risk Distribution by Count')
axes[1].set_ylabel('Number of Wallets')

plt.tight_layout()
plt.show()

## Flagged Wallets

In [ ]:
flagged = scorer.get_flagged_wallets(min_score=25)
print(f"Wallets with risk score >= 25: {len(flagged)}")

df_flagged = pd.DataFrame(flagged)
df_flagged['address_short'] = df_flagged['address'].str[:10] + '...'
df_flagged.head(20)

## Risk Score Breakdown

In [ ]:
scores = scorer.calculate_all_scores()

df_scores = pd.DataFrame([
    {
        'address': s.wallet_address,
        'total_score': s.total_score,
        'mixer_score': s.mixer_score,
        'bridge_score': s.bridge_score,
        'sybil_score': s.sybil_score,
        'layering_score': s.layering_score
    }
    for s in scores[:50]
])

df_scores.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

df_scores.set_index('address').plot(
    kind='barh',
    stacked=True,
    ax=ax,
    color=['#e74c3c', '#3498db', '#9b59b6', '#f39c12']
)
ax.set_title('Risk Score Breakdown by Category')
ax.set_xlabel('Score')
ax.set_ylabel('Wallet')
ax.legend(['Mixer', 'Bridge', 'Sybil', 'Layering'], loc='lower right')
ax.set_xlim(0, 100)

plt.tight_layout()
plt.show()

## Flag Type Analysis

In [ ]:
from collections import Counter

all_flags = []
for wallet in flagged:
    for flag in wallet.get('flags', []):
        all_flags.append(flag['type'])

flag_counts = Counter(all_flags)
print("=== Detection Flags Summary ===")
for flag_type, count in flag_counts.most_common():
    print(f"{flag_type}: {count}")

In [ ]:
flag_df = pd.DataFrame({
    'Flag Type': list(flag_counts.keys()),
    'Count': list(flag_counts.values())
}).sort_values('Count', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(flag_df['Flag Type'], flag_df['Count'], color='steelblue')
ax.set_title('Detection Flags Distribution')
ax.set_xlabel('Count')

for i, v in enumerate(flag_df['Count']):
    ax.text(v + 0.1, i, str(v), va='center')

plt.tight_layout()
plt.show()

## Critical Risk Wallets

In [ ]:
critical_wallets = [w for w in flagged if w['risk_score'] >= 75]
print(f"Critical Risk Wallets (score >= 75): {len(critical_wallets)}")

for wallet in critical_wallets:
    print(f"\n{'='*60}")
    print(f"Address: {wallet['address']}")
    print(f"Risk Score: {wallet['risk_score']:.1f}")
    print(f"Flags:")
    for flag in wallet.get('flags', []):
        print(f"  - {flag['type']}: {flag['confidence']:.0f}%")

## Export Data

In [ ]:
df_scores.to_csv('../reports/risk_scores.csv', index=False)
df_flagged.to_csv('../reports/flagged_wallets.csv', index=False)
print("Data exported to ../reports/")

In [ ]:
store.close()
print("Connection closed")